In [ ]:
!pip install -q wandb accelerate

In [ ]:
import os

WANDB_KEY = os.environ.get('WANDB_KEY')
if WANDB_KEY is not None:
    os.system(f"wandb login {WANDB_KEY}")

base_path = '/kaggle/working'
project_name = 'weight-vs-loss-convergence-pinn'
project_root = os.path.join(base_path, project_name)

if not os.path.exists(project_root):
    os.chdir(base_path)
    !git clone https://github.com/nthday-jpg/weight-vs-loss-convergence-pinn.git
    print("Repository cloned successfully!")
else:
    os.chdir(project_root)
    !git pull
    print("Repository updated successfully!")
%cd {base_path}

# Ensure project root is in PYTHONPATH for script's imports
os.environ['PYTHONPATH'] = project_root

In [ ]:
# Generate data
save_dir = os.path.join(project_root, 'data')
!python -m burgers.data.gen_data \
        --save_dir {save_dir} \
        --nn 511\
        --steps 200 \
        --nu 0.01\
        --t_final 1.0

In [ ]:
import json

config = {
    'layers': [2, 64, 1],
    'learning_rate': 1e-3,
    'num_epochs': 10000,
    'batch_size': 64,
    'l2_reg': 1e-6,
    'data_path': save_dir,
}

# Save config to JSON file
config_path = os.path.join(project_root, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

!accelerate launch burgers/train.py --config_file {config_path}